In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: 10
Connected to future database: 10


In [3]:
import pickle
import os

print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 ")
print("================================================================================")

# ================================================================================
# KONFIGURASI 1: LOAD MASING-MASING FILE PICKLE (TETAP TERPISAH)
# ================================================================================
data_cimut = {}
data_afrida = {}
data_hanif = {}

# 1. Load File cimut (Ganti nama file sesuai punyamu)
try:
    with open('fase_2_cimut.pkl', 'rb') as f:
        data_cimut = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik cimut.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_2_afrida.pkl', 'rb') as f:
        data_afrida = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Afrida.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif
try:
    with open('fase_2_hanif.pkl', 'rb') as f:
        data_hanif = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Hanif.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Hanif: {e}")

print("\n================================================================================")
# ================================================================================
# KONFIGURASI 2: ISI DAFTAR TABEL MILIK MASING-MASING ORANG
# ================================================================================
list_table_cimut = [
    "karyawan",
    "keluarga_karyawan",
    "bidang_kategori",
    "bidang_link",
]

list_table_afrida = [
    "periode",
    "parameter_nilai",
]

list_table_hanif = [
    
]

# ================================================================================
# KONFIGURASI 3: ATUR URUTAN MUTLAK PENYUNTIKAN KE DATABASE (MASTER ORDER)
# ================================================================================
# Masukkan nama tabel yang mau di-insert sesuai urutan FK (Foreign Key).
# Kamu bebas menyilangkan nama tabel di sini, sistem akan otomatis mencari pemiliknya.
master_urutan_insert = [
    "karyawan",
    "keluarga_karyawan",
    "bidang_kategori",
    "bidang_link",
    "periode",
    "parameter_nilai",
]

# ================================================================================
# SISTEM DETEKTIF: MENCARI DAN MENGGABUNGKAN DATA BERDASARKAN PEMILIKNYA
# ================================================================================
print("🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...\n")

data_siap_insert = {}

for table in master_urutan_insert:
    if table in list_table_cimut:
        data_siap_insert[table] = data_cimut.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data cimut.")
        
    elif table in list_table_afrida:
        data_siap_insert[table] = data_afrida.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Afrida.")
        
    elif table in list_table_hanif:
        data_siap_insert[table] = data_hanif.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Hanif.")
        
    else:
        # Jika kamu memasukkan nama tabel di master_urutan tapi lupa memasukkannya di list pemilik
        data_siap_insert[table] = None
        print(f"  ❌ ERROR: Tabel '{table}' tidak ada di list cimut, Afrida, maupun Hanif!")

print("\n✅ Pemetaan selesai! Data siap disuntikkan ke database.")

 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 
✓ Berhasil memuat data PKL milik cimut.
✓ Berhasil memuat data PKL milik Afrida.
✓ Berhasil memuat data PKL milik Hanif.

🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...

  📦 Tabel 'karyawan' otomatis dipetakan dari data cimut.
  📦 Tabel 'keluarga_karyawan' otomatis dipetakan dari data cimut.
  📦 Tabel 'bidang_kategori' otomatis dipetakan dari data cimut.
  📦 Tabel 'bidang_link' otomatis dipetakan dari data cimut.
  📦 Tabel 'periode' otomatis dipetakan dari data Afrida.
  📦 Tabel 'parameter_nilai' otomatis dipetakan dari data Afrida.

✅ Pemetaan selesai! Data siap disuntikkan ke database.


## Hide code

In [4]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT (ANTI SILENT-KILLER, AUTO-BATCHING & DIAGNOSTIC)
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list, batch_size=2000):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DENGAN CHUNKING (LOOPING AMAN)
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {'status': 'not_found', 'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl', 'warnings': []}
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {'status': 'empty', 'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)', 'warnings': []}
            continue
            
        try:
            # Bersihkan kolom kosong murni
            df_to_push = df_target.dropna(axis=1, how='all')
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            raw_numpy_list = df_to_push.to_numpy().tolist()
            
            # TUPLE GENERATOR (Mempertahankan "" untuk kolom Varchar NOT NULL)
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            total_rows = len(clean_data_tuples)
            actual_inserted_total = 0
            db_warnings = []
            
            # 🔥 SISTEM AUTO-BATCHING (CHUNKING) 🔥
            # Loop memotong data menjadi bagian-bagian kecil agar MySQL tidak tersedak
            for i in range(0, total_rows, batch_size):
                chunk = clean_data_tuples[i : i + batch_size]
                cursor.executemany(insert_query, chunk)
                
                # Hitung data yang berhasil masuk pada batch ini
                chunk_inserted = max(0, cursor.rowcount)
                actual_inserted_total += chunk_inserted
                
                # Jika ada yang ter-skip di batch ini, tangkap errornya (maksimal simpan 3 per tabel)
                if chunk_inserted < len(chunk) and len(db_warnings) < 3:
                    cursor.execute("SHOW WARNINGS")
                    warnings_fetched = cursor.fetchall()
                    if warnings_fetched:
                        for w in warnings_fetched:
                            w_msg = f"MySQL Warning: {w['Message']}"
                            if w_msg not in db_warnings:
                                db_warnings.append(w_msg)
                            if len(db_warnings) >= 3:
                                break
                                
                # Commit per batch agar memori stabil
                db_connection.commit()
            
            # Evaluasi Status Akhir Tabel
            if actual_inserted_total == total_rows:
                status_flag = 'success'
                msg = f'✓ {table_name}: SEMPURNA! {total_rows}/{total_rows} baris sukses masuk database.'
            else:
                status_flag = 'partial_warning'
                msg = f'⚠️ {table_name}: TER-SKIP! Dikirim {total_rows} baris, tapi yang masuk DB HANYA {actual_inserted_total} baris.'

            results[table_name] = {
                'status': status_flag, 
                'msg': msg,
                'warnings': db_warnings
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'msg': f'✗ {table_name}: Gagal total saat eksekusi insert - Alasan: {e}',
                'warnings': []
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    print("🟢 TABEL YANG 100% SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') == 'success':
            print(f"  {res['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses sempurna)")

    print("\n🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):")
    failed_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') in ['failed', 'partial_warning', 'not_found', 'empty']:
            print(f"  {res['msg']}")
            
            # Cetak alasan dari MySQL (Dibatasi 3 agar tidak merusak tampilan Jupyter)
            if res.get('warnings'):
                for w_msg in res['warnings']:
                    print(f"      -> 🕵️ {w_msg}")
                    
            failed_exist = True
            
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.")
            
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            res = results[table_name]
            
            if res['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(3))
                print("-" * 80)
                
            elif res['status'] in ['failed', 'partial_warning']:
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Pesan Sistem: {res['msg']}")
                print("-" * 50)
                print("Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data Pandas untuk tabel '{table_name}':")
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

## Output

In [5]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL
# ================================================================================
results_fase_2 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=data_siap_insert,       # <--- Menggunakan data yang sudah di-mapping otomatis
    ordered_list=master_urutan_insert   # <--- Menggunakan urutan master buatanmu
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG 100% SUKSES MASUK:
  ✓ karyawan: SEMPURNA! 51/51 baris sukses masuk database.
  ✓ keluarga_karyawan: SEMPURNA! 64/64 baris sukses masuk database.
  ✓ bidang_kategori: SEMPURNA! 12/12 baris sukses masuk database.
  ✓ bidang_link: SEMPURNA! 7/7 baris sukses masuk database.
  ✓ periode: SEMPURNA! 91/91 baris sukses masuk database.
  ✓ parameter_nilai: SEMPURNA! 1187/1187 baris sukses masuk database.

🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):
  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.

📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL

📂 [🟢 PREVIEW TABEL SUKSES: KARYAWAN]
--------------------------------------------------


,id_karyawan,id_user,kode_karyawan,nik_ktp,nama_lengkap,nama_panggilan,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,...,akun_instagram,akun_facebook,link_dokumen_pribadi,riwayat_kesehatan,tanggal_bergabung,keahlian,id_shift,status_aktif,foto_profile,ttd_digital
0,1,U00001,LEAP00102VI23,None,ADMINISTRATOR,ADMINISTRATOR,None,None,Perempuan,None,...,None,None,None,None,2026-01-01,None,2,1,logo.png,None
1,2,U00003,LEAP00313III23,3514186411980002,"Graciela Evanda Ronadi, S.Kom.",Graciela,Sidoarjo,1998-11-24,Perempuan,None,...,https://www.instagram.com/gracielaevr/,None,https://drive.google.com/drive/folders/1alw9Su...,"Maag, tipes",2026-01-01,None,2,1,1707279559_ec864cc58f50e8b890d5.jpg,1702002184_d82747bcbd0f2bdab0a7.png
2,3,U00011,LEAP01101XII20,3515135105910001,DANIAR AULIA RIZKI,DANIAR,SURABAYA,1991-05-11,Perempuan,B,...,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va...,PREKLAMSIA,2026-01-01,None,2,0,1692700456_2d6352eb557d734e53b7.jpeg,None


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KELUARGA_KARYAWAN]
--------------------------------------------------


,id_keluarga,id_karyawan,hubungan_keluarga,nama_lengkap,pekerjaan,nomor_hp
0,None,2,Ibu,Nur Fatmawati,Wiraswasta,081234477137
1,None,9,Suami,Fathul Kirom,Suami,0852-3101-7799
2,None,12,Ayah,"Suwandi, S.Pd.",Guru Matematika SMAN 17 Surabaya,087853591616


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: BIDANG_KATEGORI]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PERIODE]
--------------------------------------------------


,id_periode,nama_periode,id_kursus,jumlah_sesi,tahun_ajar,tanggal_mulai,status,is_active
0,P00006,General English Term I July-October 2023,K00001,30,2023/2024,2023-07-04,1,1
1,P00008,General English Term II Oct '23 - Feb '24,K00001,30,2023/2024,2023-10-25,1,1
2,P00009,General English Term III Feb-Jun 2024,K00001,30,2023/2024,2024-02-21,1,1


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PARAMETER_NILAI]
--------------------------------------------------


,id_level,nama_parameter,status_parameter
0,L00022,Class participation,0
1,L00022,Oral,0
2,L00022,Listening,0


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [6]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 1 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_2 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )